# Web Scraping Demo

This notebook implements the sample shown on slides 16 and 17. It serves `sample_table.html` over a local HTTP connection, downloads the page with `requests`, parses the table with BeautifulSoup, and creates a pandas DataFrame.

In [1]:
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from threading import Thread

import pandas as pd
import requests
from bs4 import BeautifulSoup

## Start a local web server

A browser normally retrieves a page over HTTP. The following cell starts a temporary local server so the demo follows that same request flow without relying on an external website.

In [2]:
candidates = [Path.cwd(), Path.cwd() / 'Real World Data Wrangling' / 'web_scraping_demo']
demo_dir = next((path for path in candidates if (path / 'sample_table.html').exists()), None)
if demo_dir is None:
    raise FileNotFoundError('Could not find sample_table.html. Run this notebook from the repository or demo directory.')

handler = partial(SimpleHTTPRequestHandler, directory=str(demo_dir))
server = ThreadingHTTPServer(('127.0.0.1', 0), handler)
server_thread = Thread(target=server.serve_forever, daemon=True)
server_thread.start()

url = f'http://127.0.0.1:{server.server_port}/sample_table.html'
print(f'Serving the sample page at {url}')

Serving the sample page at http://127.0.0.1:38121/sample_table.html


127.0.0.1 - - [14/Jul/2026 17:04:18] "GET /sample_table.html HTTP/1.1" 200 -
127.0.0.1 - - [14/Jul/2026 17:04:18] code 404, message File not found
127.0.0.1 - - [14/Jul/2026 17:04:18] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [14/Jul/2026 17:04:34] "GET /sample_table.html HTTP/1.1" 200 -


## Request and parse the page

In [3]:
response = requests.get(url, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.content, 'html.parser')
table = soup.find('table', id='people-table')
if table is None:
    raise ValueError('The expected table was not found on the page.')

headers = [th.get_text(strip=True) for th in table.select('thead th')]
rows = [
    [td.get_text(strip=True) for td in row.select('td')]
    for row in table.select('tbody tr')
]

if not headers or any(len(row) != len(headers) for row in rows):
    raise ValueError('The table headers and rows do not have a consistent structure.')

df = pd.DataFrame(rows, columns=headers)
df['Age'] = pd.to_numeric(df['Age'])
df

,Name,Age,City
0,Alice,30,New York
1,Bob,25,Los Angeles
2,Charlie,35,Chicago


## Verify the result

In [4]:
assert df.columns.tolist() == ['Name', 'Age', 'City']
assert df.shape == (3, 3)
assert df['Age'].tolist() == [30, 25, 35]
print('Validation passed.')

Validation passed.


## Stop the local server

In [5]:
server.shutdown()
server.server_close()
server_thread.join(timeout=5)
print('Local server stopped.')

Local server stopped.
